<div style="padding: 60px;
  text-align: center;
  background: #7dc8ca;
  color: #003049;
  font-size: 20px;">
  <h2>Inclass: Unsupervised Learning</h2>
   <hr>
</div>


- **This notebook based on main material**
- **Course Length**: 6 hours
- **Instructor** : Dyah Nurlita
- **Last Updated**: October 2025

## Training Objectives

This coursebook is intended for participants who have completed the preceding courses. 

The coursebook focuses on:

* **Introduction to Unsupervised Learning**
    * Supervised vs Unsupervised Learning
* **K-Means Clustering**
    * Understanding and Practicing K-Means
    * Optimal Number of Cluster
* **K-Medoids Clustering**
    * Understanding and Practicing K-Medoids
    * Choosing Right Clustering Method based on Data

# 🤖 Introduction to Unsupervised Learning

Machine Learning terbagi menjadi dua jenis utama: **Supervised Learning** dan **Unsupervised Learning**. Untuk memahami posisi *unsupervised learning*, mari kita bandingkan dulu dengan supervised learning.

## 🎯 Supervised vs Unsupervised Learning

* **Supervised Learning**

  * Ada **label/target** (misalnya: apakah email itu *spam* atau *bukan spam*).
  * Tujuan: melatih model agar bisa memprediksi label baru berdasarkan data yang belum pernah dilihat.

* **Unsupervised Learning**

  * **Tidak ada target/label**.
  * Tujuan: menemukan pola atau struktur tersembunyi dalam data.
  * Contoh: mengelompokkan pelanggan berdasarkan kebiasaan belanja, tanpa tahu sebelumnya kelompok apa yang ada.

👉 Singkatnya:

* **Supervised** → ada label, belajar dengan “jawaban”.
* **Unsupervised** → tidak ada label, belajar mencari pola sendiri.

---

## 🔍 Unsupervised Learning

**Karakteristik utama:**

* ❌ Tidak memiliki target variabel.
* 🧩 Digunakan untuk menemukan pola atau struktur dalam data.
* 📊 Umumnya dipakai pada tahap **Exploratory Data Analysis (EDA)** atau **pre-processing**.
* 🤔 Evaluasi lebih sulit, karena tidak ada *ground truth*.

**Contoh penerapan:**

* *Clustering*: Mengelompokkan pelanggan, dokumen, atau gambar.
* *Dimensionality Reduction*: Seperti PCA (Principal Component Analysis), untuk menyederhanakan data tanpa kehilangan informasi penting.

Dalam modul ini, kita akan fokus pada metode **Clustering**:

* K-Means Clustering
* K-Medoids Clustering

Keduanya banyak digunakan di dunia nyata, khususnya untuk segmentasi data.



In [ ]:
import importlib
from pylab import rcParams
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# for data manipulation
import pandas as pd 
pd.set_option('display.float_format', lambda x: '%.3f' % x) # suppress scientific notation
pd.set_option('display.max_columns', None) # display all columns

# for matrix-based computation
import numpy as np

# data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as colors

# Scalers
from sklearn.preprocessing import StandardScaler

# clustering
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids

# Gower matrix
import gower as go

# custom functions
from helper import plot_elbow_kmeans, plot_elbow_kmedoids, plot_depth_readings_cluster, biplot_kmeans

%matplotlib inline

## 🛢️ Study Case: Clustering on Well Logging Data

Dalam industri minyak dan gas, **sumur** adalah komponen utama untuk mengekstraksi sumber daya dari bawah permukaan bumi. Saat pengeboran berlangsung, perusahaan perlu menentukan apakah sumur yang sedang digali benar-benar mengandung minyak atau gas yang bernilai ekonomi.

Salah satu cara untuk mengetahuinya adalah melalui **well logging** — yaitu memasukkan alat logging ke dalam sumur untuk mengukur berbagai sifat batuan pada kedalaman tertentu.

![](assets/well_logging.jpg)

### 🔧 Apa itu Well Logging?

* Struktur alat logging menyerupai pipa dengan berbagai sensor di dalamnya.
* Setiap sensor menghasilkan **data pengukuran** (misalnya resistivitas, densitas, porositas).
* Data ini kemudian dianalisis oleh geologis untuk menilai apakah pengeboran layak dilanjutkan ke tahap berikutnya: **well completion** (penyelesaian sumur agar siap berproduksi).

> 👉 Intinya, well logging membantu menjawab: *“Apakah batuan di sekitar sumur ini mengandung cukup minyak/gas?”*


### 📊 Dataset yang Digunakan

Untuk latihan, kita akan menggunakan dataset dari **[FORCE 2020 Well log and lithofacies dataset for machine learning competition](https://zenodo.org/records/4351156)**.

* Dataset ini berisi data logging dari **118 sumur di Laut Norwegia**.
* Data ini sebelumnya digunakan untuk kompetisi ML dengan tujuan memprediksi litofasies (jenis lapisan batuan) berdasarkan data seismik dan logging.
* Banyak nilai **missing value** ditemukan di dataset asli. Untuk modul ini, kita sudah melakukan:

  * Prapemrosesan
  * Sampling data dari sumur **ID `25/2-7`**
  * Mengganti nilai hilang dengan `0` atau nilai statistik lain.


### 🎯 Tujuan Analisis

Dalam kasus ini, kita ingin menjawab pertanyaan berikut:

1. Apakah sumur yang sedang digali memiliki **potensi penambangan** yang cukup?
2. Karakteristik atau variabel apa yang paling berpengaruh terhadap potensi tersebut?
3. Bisakah kita mengelompokkan jenis batuan (**lithologi**) hanya dari data alat logging?

💡 Metode yang digunakan: **Clustering**.

### 1. 📂 Load Data

In [ ]:
# load data
well_2527 = 

# 5 first observations


#### **📑 Deskripsi Variabel pada Well Logging Data**

Dataset ini terdiri dari beberapa kelompok kolom:

**🗂️ Metadata Columns**

* `WELL` → Nama sumur.
* `DEPTH_MD` → Kedalaman pengeboran yang diukur (*measured depth*, dalam meter).
* `X_LOC` → Koordinat X (UTM).
* `Y_LOC` → Koordinat Y (UTM).
* `Z_LOC` → Kedalaman (meter).
* `GROUP` → Grup litostratigrafi menurut NPD.
* `FORMATION` → Formasi litostratigrafi menurut NPD.

**📉 Well Log Curves (hasil pengukuran alat logging)**

* `BS` → Borehole size (inch).
* `CALI` → Caliper log (inch).
* `RHOB` → Bulk density log (g/cm³).
* `GR` → Gamma ray log (satuan API).
* `SGR` → Spectra gamma ray log (satuan API).
* `ROP` → Rate of penetration (m/h).
* `NPHI` → Neutron porosity log (fraksi → bisa ditafsirkan dalam persen).
* `PEF` → Photoelectric absorption factor log (barns/electron).
* `RSHA` → Shallow resistivity measurement (Ω·m).
* `RMED` → Medium resistivity measurement (Ω·m).
* `RDEP` → Deep resistivity measurement (Ω·m).
* `DTS` → Shear wave sonic log (µs/ft).
* `DTC` → Compressional wave sonic log (µs/ft).

**🏷️ Interpretation Columns**

* `LITHOFACIES_CLASS` → Label litologi (dari interpretasi ahli geologi).
* `LITHOFACIES_CLASS_CONFIDENCE` → Tingkat kepercayaan interpretasi (1 = tinggi, 2 = sedang, 3 = rendah).

> 💡 **Catatan penting:**
> 
> Walaupun dataset sudah memiliki kolom label (`LITHOFACIES_CLASS`), tujuan kita adalah **menggunakan unsupervised learning** untuk mengelompokkan litofasies berdasarkan data logging. Nantinya, hasil cluster bisa dibandingkan dengan label asli untuk evaluasi.

### 2. Mempersiapkan Data

Dalam proses persiapan data, ada beberapa langkah yang akan dilakukan untuk memastikan data kita siap untuk dimodelkan. Beberapa langkah ini termasuk melihat informasi dalam data, memeriksa nilai yang hilang, dan mengidentifikasi duplikat.

In [ ]:
# the information in the data


In [ ]:
# check for missing values


In [ ]:
# check for duplicates


> Insight: 

### 3. ⚙️ Data Preprocessing

Sebelum kita melakukan clustering, data perlu dipersiapkan agar siap dimodelkan. Ada beberapa langkah utama:

1. **Pilih kolom numerik** → hanya variabel log hasil pengukuran yang bertipe numerik.
2. **Scaling/Standarisasi** → menyamakan rentang nilai, karena tiap variabel punya skala berbeda (misalnya `GR` dalam API vs `RHOB` dalam g/cm³).
3. **Pemeriksaan data** → pastikan tidak ada duplikat, nilai yang hilang sudah ditangani, dan format sesuai.

#### 🔢 Memilih Kolom Numerik

Kita mulai dengan memfilter hanya variabel numerik yang relevan untuk clustering.

In [ ]:
# cek persebaran data dengan describe()


In [ ]:
# 🔎 Pilih kolom numerik dari dataset
well_num = 

In [ ]:
# Cek dimensi data hasil filter
print("Dimensi data numerik:", well_num.shape)

<u>**Memilih kolom yang relevan**</u>

In [ ]:
# pilih kolom yang relevan untuk pemodelan
well_num_selected = 

#### ⚖️ Melakukan Scaling

🔎 Kenapa Scaling Penting?

Tujuan scaling adalah **menyamakan rentang nilai antar variabel** agar tidak ada variabel tertentu yang mendominasi analisis hanya karena skalanya lebih besar.

* **Variance dan covariance dipengaruhi skala data.**
  Misalnya, `DEPTH_MD` (kedalaman dalam meter) bisa bernilai ribuan, sedangkan `NPHI` (porositas) hanya bernilai 0–1.
* Jika tidak di-scale, variabel berskala besar akan memberikan kontribusi berlebih pada analisis seperti **clustering**.

In [ ]:
# Mengecek rentang nilai tiap kolom


> ......

#### ⚙️ Implementasi StandardScaler

Kita gunakan `StandardScaler` dari scikit-learn.

* **`fit()`** → menghitung mean dan standard deviation.
* **`transform()`** → menstandardisasi data.
* **`fit_transform()`** → menggabungkan keduanya dalam satu langkah.

In [ ]:
# Import library
from sklearn.preprocessing import StandardScaler

# Copy data numerik
well_num_scaled = well_num_selected.copy()

# Ambil nama kolom numerik
cols = 

# Buat objek scaler
scaler = 

# Fit dan transformasi sekaligus


In [ ]:
# Cek hasil setelah scaling
well_num_scaled.describe()

---

## 🧩 Clustering

Setelah data kita eksplorasi dan diproses, sekarang saatnya melakukan analisis lebih lanjut menggunakan **clustering**.

👉 **Clustering** adalah proses mengelompokkan data berdasarkan kemiripan karakteristiknya.

* Data dalam **satu cluster** → cenderung memiliki karakteristik yang mirip.
* Data dari **cluster berbeda** → cenderung memiliki karakteristik yang berbeda.

Ada banyak metode clustering, namun dalam modul ini kita akan fokus pada:

* **K-Means Clustering**
* **K-Medoids Clustering**

Keduanya sama-sama berbasis *centroid* (titik pusat), tetapi ada perbedaan dalam cara menentukan representasi cluster.

### 🔵 K-Means Clustering

#### 📖 Konsep Dasar

K-Means adalah salah satu algoritma clustering yang paling populer dan sederhana.

* Disebut *K-Means* karena jumlah cluster **K** ditentukan oleh pengguna, dan pusat cluster dihitung menggunakan **mean** dari anggota cluster.
* Termasuk kategori **centroid-based clustering algorithm** → artinya tiap cluster direpresentasikan oleh titik pusat (*centroid*).

#### ⚙️ Proses Algoritma K-Means

1. **Random Initialization**
   Meletakkan **K centroid** secara acak di ruang data.

2. **Cluster Assignment**
   Setiap observasi dialokasikan ke cluster terdekat, berdasarkan jarak (biasanya **Euclidean distance**).

   > Euclidean distance = jarak garis lurus antara dua titik.

3. **Centroid Update**
   Geser posisi centroid ke rata-rata (mean) dari seluruh anggota cluster tersebut.

4. **Repeat**
   Ulangi langkah 2 dan 3 hingga posisi centroid stabil (tidak banyak berubah) atau tidak ada observasi yang berpindah cluster lagi.

👉 Jumlah cluster **K** harus ditentukan sebelum menjalankan algoritma.

<img src="assets/kmeans.jpg" width="700">  

> 💡 **Tips Belajar**:
> Kalau masih bingung dengan konsep K-Means, coba lihat visualisasi interaktif di sini:
> 🔗 [Visualizing K-Means Clustering](https://www.naftaliharris.com/blog/visualizing-k-means-clustering/)

### 🔄 K-Means Workflow

Proses penerapan **K-Means Clustering** biasanya melalui 4 tahap utama:

1. 🧹 **Persiapan Data**
2. 🔢 **Menentukan Nilai K**
3. 🏗️ **Membuat Cluster**
4. 🧭 **Cluster Profiling**

#### 1️⃣ Mempersiapkan Data

Pada tahap ini kita memastikan:

* Semua kolom yang dipakai sudah berupa **tipe data numerik**
* Data sudah melalui proses **scaling/standardisasi**

👉 Dua langkah ini sudah kita lakukan di bagian sebelumnya.

In [ ]:
# Mengecek data
well_num_scaled.describe()

#### 2️⃣ Menentukan Nilai `K`

Menentukan jumlah cluster (`k`) adalah bagian paling penting. Ada dua pendekatan:

**🔹 a. Berdasarkan Kebutuhan Bisnis**

* Jumlah cluster bisa ditentukan sesuai konteks bisnis atau tujuan analisis.
* Misalnya: pada kasus well logging, kita bisa mencoba menggunakan jumlah **kelas litologi unik** pada kolom `LITHOLOGY_CLASS`.
* Ini bisa menjadi *benchmark* untuk membandingkan apakah clustering mendekati hasil interpretasi geologis.

> **Catatan**: pada praktik unsupervised sebenarnya *ground truth* ini tidak selalu tersedia.

In [ ]:
# cek jumlah nilai unik


**🔹 b. Berdasarkan Metode Elbow**

* Jalankan K-Means dengan berbagai nilai `k`
* Plot hasil *within-cluster sum of squares (WCSS)*
* Pilih titik “patah” (elbow) → saat grafik mulai melandai.

📌 Untuk contoh awal, kita gunakan **pendekatan kebutuhan bisnis**.

#### 3️⃣ Modelling dengan `k = 8`

Misalnya, kita diminta membuat cluster **jenis batuan** dengan `k = 8`.
Kita bisa menggunakan fungsi `KMeans()` dari `sklearn.cluster`:

* `n_clusters=` → jumlah cluster yang ingin dibuat
* `random_state=` → angka pengunci agar hasil random konsisten

Seperti permodelan lainnya, fungsi `KMeans()` akan dilanjutkan dengan fungsi `.fit()` untuk melakukan pembuatan cluster dari data.

> Dokumentasi detail: [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)

In [ ]:
# Import Library
from sklearn.cluster import KMeans

In [ ]:
# Buat model dengan k = 8
kmeans_lith8 = 

# Fit model ke data scaled


#### 4️⃣ Melihat Hasil Cluster

Label cluster untuk setiap observasi bisa diperoleh dari atribut `.labels_`.

In [ ]:
# Ambil label cluster
label8 = 

# Tampilkan 30 label pertama


💡 **Insight:**

* .....
* .....

#### 📊 Hasil dan Evaluasi Cluster

Ingat kembali bahwa algoritma clustering masuk ke dalam algoritma **Unsupervised Learning**, sehingga tidak terdapat *ground truth* untuk hasil evaluasi modelnya.

⚖️ **Goodness of Fit** yang biasanya dipakai pada algoritma clustering yaitu:

* 🔹 **Evaluasi WSS (Within Sum of Square)**
  
  - Nilai ini menunjukkan kemiripan antar anggota pada cluster.
  - 👉 **Semakin kecil nilai WSS maka semakin baik**, karena mengindikasikan bahwa anggota pada cluster semakin mirip satu sama lain.

  - 📐 Rumus singkat:
    - `Hitung jarak tiap observasi ke centroid tiap cluster -> dikuadratkan -> dijumlahkan`
  - Untuk melihat nilai evaluasi WSS, kita akan menggunakan atributes `.inertia_` dari hasil pembuatan cluster.

In [ ]:
WSS = 
print('WSS K Means:', WSS)

* 🔹 **Evaluasi BSS (Between Sum of Square)**
  - Nilai ini menunjukkan jarak antar centroid cluster.
  - 👉 **Semakin besar nilai BSS semakin baik**, karena mengindikasikan bahwa antar cluster memiliki karakteristik berbeda.
  - 📐 Rumus singkat:
    - `Hitung jarak centroid tiap cluster ke global sample mean -> dikuadratkan -> dikali jumlah data tiap cluster -> dijumlahkan`
  - `BCSS(data, object_cluster)`

In [ ]:
# BSS
from helper import BCSS
bss_lith8 = 

print('BSS K Means:', bss_lith8)

* 🔹 **Evaluasi TSS (Total Sum of Square)**
  Jarak tiap observasi ke global sample mean.
  
  Rumus:

  $$
  TSS = BSS + WSS
  $$

  Dari sini kita bisa hitung **Explained Variance (EV)** ✨
  
  👉 Semakin mendekati **100% semakin baik**, karena cluster semakin mewakili persebaran data.

  $$
  EV = \frac{BSS}{TSS} 
  $$

<!-- import numpy as np
import pandas as pd

def compute_tss(dataframe):
    # Step 1: Calculate the mean of the entire dataset
    mean_data = dataframe.mean()

    # Step 2: Compute the sum of squared distances
    tss = np.sum(np.linalg.norm(dataframe - mean_data, axis=1)**2)

    return tss

customer_scale.mean()

# Example usage:
# Replace 'your_data' with your actual dataset
your_data = customer_scale
tss_value = compute_tss(your_data)

print("Total Sum of Squares (TSS):", tss_value) -->

In [ ]:
# TSS = BSS + WSS
tss_lith8 = 

print('TSS K Means:', tss_lith8)

In [ ]:
# Explained variance
EV = 
print('Explained Variance K Means:', EV)

💡 **Insight** : Informasi yang berhasil ditangkap oleh 8 cluster adalah sebesar **....**

---

#### 📉 Dari Evaluasi ke Pemilihan Nilai `k`

Sampai di tahap sebelumnya, kita sudah melakukan evaluasi hasil clustering menggunakan **WSS**, **BSS**, dan **Explained Variance (EV)**. Dari hasil itu, terlihat bahwa pemilihan jumlah cluster (`k`) memang sangat berpengaruh terhadap kualitas pemisahan data.

👉 Namun muncul pertanyaan:
Kalau begitu, **berapa jumlah cluster yang sebaiknya kita pilih?**

Jika kita hanya mengandalkan kebutuhan bisnis saja, memang seringkali ada angka yang jelas (misalnya 8 kelompok litologi). Tapi, dalam banyak kasus dunia nyata, kita **tidak selalu tahu berapa `k` yang ideal**.

Di sinilah kita butuh pendekatan objektif menggunakan **Elbow Plot**.


#### 📊 *Modelling* dengan Pemilihan `k` Optimum (*Elbow Plot*)

Semakin tinggi `k`:

* 🔽 WSS semakin mendekati 0
* 🔼 BSS semakin mendekati TSS (atau BSS/TSS mendekati 1)

Kalau begitu apakah kita selalu memilih `k = jumlah observasi`? 🤔


Tentu tidak. Ada 2 pertimbangan utama dalam menentukan nilai `k`:

1. 💼 **Kebutuhan bisnis** → misalnya jumlah segmen pelanggan, jenis litologi, atau kategori yang memang ingin dipetakan.
2. 📐 **Pendekatan objektif** → menggunakan metode *Elbow*, untuk mencari titik optimal.

Metode *Elbow* bekerja dengan cara memplot:

* Sumbu **X** → nilai `k` (jumlah cluster)
* Sumbu **Y** → nilai WSS (variansi dalam cluster)

🔎 Nilai `k` optimum terlihat pada titik "patah" (*elbow*) di mana penurunan WSS mulai melandai.

In [ ]:
# Membuat visualisasi elbow untuk memilih k optimum


👉 Pilih nilai `k` dengan kondisi: ketika `k` ditambah, **penurunan WSS tidak lagi signifikan** (garis mulai datar).

> 💡 Berdasarkan grafik elbow tersebut, manakah nilai `k` maksimum yang dapat dipilih? … cluster.

### 🧩 *Cluster Profiling*

Tahap terakhir dari proses clustering adalah **profiling**, yaitu memahami karakteristik dari masing-masing cluster yang sudah terbentuk. Tujuan dari tahap ini adalah:

* 🔎 Mengetahui apa yang membedakan tiap cluster
* 📝 Memberi interpretasi praktis agar hasil clustering bisa dipakai dalam pengambilan keputusan

👉 Dalam konteks ini, kita akan mempelajari hasil cluster berdasarkan properti kedalaman dan pembacaan alat logging.

#### 📂 Menambahkan Label Cluster ke Data Asli

Pertama, mari kita salin kembali data asli lalu menambahkan label hasil clustering dari model *K-means*.

In [ ]:
# Salin data tanpa kolom yang tidak diperlukan: 'GROUP','FORMATION','LITHOLOGY_CLASS_CONFIDENCE'
well_2527_copy = 

Selanjutnya, kita akan menambahkan label dari model K-means sebelumnya ke dataframe `well_2527_copy`.

In [ ]:
# Assign label hasil K-means (k=8)
well_2527_copy['LABEL_K8'] = 

In [ ]:
# sanity check


#### ⛏️ Profiling Berdasarkan Kedalaman (`DEPTH_MD`)


Selanjutnya, kita dapat melakukan profiling berdasarkan cluster yang terbentuk. Ada berbagai cara untuk melakukan hal ini. Sebagai contoh, kita dapat melakukan agregasi data berdasarkan cluster. Kita akan membuat profil cluster yang terbentuk dari model K-means dengan $k = 8$, yang diberi nama `kmeans_lith8`.

Kita bisa melihat **median kedalaman** untuk tiap cluster, lalu membandingkannya.

In [ ]:
# 1st: look from depth
well_2527_copy

🔎 Dari hasil ini, kita bisa melihat cluster mana yang cenderung berada di dekat permukaan, dan mana yang berada lebih dalam.

> Insight:
>
> * .....
> * .....
> * .....
> * .....

#### 🛢️ Profiling Berdasarkan Porositas Neutron (NPHI)

Selanjutnya, mari kita analisis cluster berdasarkan nilai `NPHI` (Neutron Porosity). Variabel ini penting untuk mengukur **kandungan hidrogen**, yang sering digunakan sebagai indikator adanya minyak atau gas.

In [ ]:
# 2st: look from neutron porosity
well_2527_copy

🔎 Dari hasil ini, kita bisa melihat cluster mana yang cenderung berada di dekat permukaan, dan mana yang berada lebih dalam.

> * .....
> * .....
> * .....
> * .....

> Insight: 👉 ......




📝 Interpretasi Hasil Cluster

* **Cluster 0** → ......
* **Cluster 1** → ......
* **Cluster 2** → ......
* **Cluster 3** → ......
* **Cluster 4** → ......
* **Cluster 5** → ......
* **Cluster 6** → ......
* **Cluster 7** → ......

#### 🔧 [Additional] Kode Profiling Tambahan

Eksplor semua cluster sekaligus:

In [ ]:
# Profiling berdasarkan median untuk semua variabel numerik
profiling_result = well_2527_copy.groupby('LABEL_K8').median()

# Urutkan berdasarkan kedalaman
profiling_result.sort_values(by='DEPTH_MD', inplace=True)
print(profiling_result[['DEPTH_MD','NPHI']])

# Visualisasi ringkas
profiling_result[['DEPTH_MD','NPHI']].plot(kind='bar', subplots=True, layout=(1,2), figsize=(12,4), rot=0)

Eksplor berdasarkan range

In [ ]:
# Ringkasan statistik per cluster
summary = well_2527_copy.groupby('LABEL_K8').agg(['mean','median','min','max'])
summary[['DEPTH_MD','NPHI']]

---

## 🔹 K-Medoids / Partitioning Around Medoids (PAM)

Setelah kita memahami dan mempraktikkan metode **K-Means**, ada satu pertanyaan penting yang sering muncul:

👉 *“Bagaimana kalau data kita memiliki **outlier** atau nilai ekstrim, apakah K-Means tetap bekerja dengan baik?”*

Jawabannya: **tidak selalu.**
K-Means menggunakan *mean* sebagai pusat cluster (*centroid*), sehingga ketika ada **outlier**, posisi centroid bisa “tertarik” ke arah data ekstrim tersebut. Akibatnya, hasil clustering bisa menjadi kurang representatif.

Untuk mengatasi kelemahan ini, kita bisa beralih ke metode yang masih “keluarga dekat” dengan K-Means, yaitu **K-Medoids (atau Partitioning Around Medoids / PAM)**.

### 📌 Apa itu K-Medoids?

* Sama seperti K-Means, tujuannya adalah mengelompokkan data ke dalam **k cluster**.
* Bedanya, pusat cluster di K-Medoids bukan dihitung dari rata-rata (mean), melainkan dipilih dari **salah satu data asli** yang disebut **medoid**.
* Karena menggunakan medoid (data nyata), metode ini lebih **tahan terhadap outlier** dan bisa bekerja lebih baik pada data dengan kombinasi numerik maupun kategorikal.

### 🧩 Perbedaan K-Means vs K-Medoids

| Aspek                         | K-Means                      | K-Medoids                                     |
| ----------------------------- | ---------------------------- | --------------------------------------------- |
| Pusat cluster                 | Centroid (nilai rata-rata)   | Medoid (salah satu titik data)                |
| Sensitivitas terhadap outlier | Tinggi 🚨                    | Rendah ✅                                      |
| Jenis data                    | Numerik (Euclidean distance) | Numerik & Kategorikal (dengan Gower distance) |


### ⚙️ Bagaimana Cara Kerja K-Medoids?

1. Pilih sejumlah medoid awal secara acak.
2. Hitung jarak setiap data ke medoid terdekat.
3. Jika ada titik data yang bisa menjadi medoid baru (dan menurunkan total jarak), maka **tukar posisi medoid**.
4. Ulangi langkah 2–3 sampai posisi medoid tidak berubah lagi.

Secara matematis:

$$
S = b - a
$$

* `a` = total jarak objek ke medoid lama
* `b` = total jarak objek ke medoid baru
  Jika `S < 0`, maka lakukan pergantian medoid.

### 📊 Mengukur Jarak pada K-Medoids

Pada K-Means kita menggunakan **Euclidean Distance**.
Namun, K-Medoids lebih fleksibel, terutama ketika data kita berupa campuran numerik + kategorikal.

➡️ Solusi: gunakan **Gower Distance**

* Bisa menghitung jarak untuk data numerik, kategorikal, maupun campuran.
* Sudah tersedia di Python lewat library `gower`.

Contoh:

Metode Gower Distance tersimpan pada `library gower` dan fungsi yang dapat digunakan adalah `gower_matrix()`.

In [ ]:
import gower

# Hitung jarak antar observasi dengan Gower Distance
well2527_gower = 

print("Matriks jarak Gower:", well2527_gower.shape)

### 📝 Catatan Penting

* Jika data **murni numerik**, sebenarnya Euclidean masih bisa dipakai.
* Jika ada campuran kategorikal (misalnya formasi batuan dengan label tertentu), **Gower Distance** lebih disarankan.
* Karena K-Medoids menggunakan medoid (salah satu data aktual), hasilnya biasanya lebih **stabil dan robust** dibanding K-Means.

---

### 📊 *Modelling* dengan K-Medoids

Pada **K-Means**, kita sudah belajar bagaimana menentukan jumlah cluster `k` yang optimum.
Prinsip yang sama juga berlaku pada **K-Medoids**:

* Bisa berdasarkan **kebutuhan bisnis**
* Bisa juga menggunakan pendekatan **visualisasi elbow plot**.

#### 🔎 1. Menentukan Jumlah Cluster dengan *Elbow Plot*

Kita buat visualisasi elbow plot dari matriks jarak **Gower Distance** untuk memilih jumlah cluster terbaik.

In [ ]:
# Elbow method untuk K-Medoids

📌 Dari grafik elbow terlihat bahwa titik “patah” terjadi pada **k = ...**, sehingga jumlah cluster yang optimum adalah **... cluster**.

#### ⚙️ 2. Pemodelan dengan K-Medoids

Sekarang kita bisa membentuk model K-Medoids dengan parameter:

* `n_clusters` : jumlah cluster
* `method` : gunakan `"pam"`
* `metric` : `"precomputed"` (karena kita sudah hitung jarak dengan Gower)
* `random_state` : untuk memastikan hasil konsisten (gunakan `123`)

Setelah fungsi tersebut terisi akan kita gabungkan dengan fungsi `.fit()` dengan random_state = 123.

In [ ]:
from sklearn_extra.cluster import KMedoids

kmed_lith = 

In [ ]:
kmed_lith

📌 Untuk melihat label hasil pengelompokan, kita dapat menggunakan  attribute `.labels_` 

In [ ]:
# label hasil clustering
label_kmed = 

# tampilkan 10 label pertama
label_kmed[:10]

#### 📍 3. Medoid vs Centroid

Karena kita menggunakan **precomputed distance**, informasi tentang titik pusat cluster (medoid) tidak langsung tersedia.

Namun, kita bisa mengaksesnya melalui atribut `.medoid_indices_`:

In [ ]:
# Mendapatkan indeks medoid
medoid_indices = kmed_lith
print("Indeks medoid di data asli:", medoid_indices)

💡 *Perbedaan dengan K-Means:*

* **K-Means** punya *centroid* (hasil rata-rata, bisa berupa titik imajiner).
* **K-Medoids** punya *medoid* (salah satu data nyata di dalam cluster).

#### 📈 4. Evaluasi Goodness of Fit

Seperti di K-Means, kita bisa menghitung **Within-Cluster Sum of Squares (WSS)**, **Between-Cluster Sum of Squares (BSS)**, dan **Total Sum of Squares (TSS)**, lalu menghitung **Explained Variance (EV)**.

In [ ]:
# WSS
WSS = 
print('WSS K Medoids:', WSS)

In [ ]:
# Menghitung centroid global
global_centroid = np.mean(well2527_gower[medoid_indices, :], axis=0)

# Menghitung BSS
BSS = 0
for medoid_index in medoid_indices:
    BSS += np.sum(well2527_gower[medoid_index, :] - global_centroid) ** 2
    
print('BSS K Medoids:', BSS)

In [ ]:
# TSS
TSS = 
print('TSS K Medoids:', TSS)

In [ ]:
# Explained variance (bss/tss)
EV = 
print('Explained Variance K medoids:', EV)

> 📌 Hasil: dengan **8 cluster**, metode K-Medoids berhasil menjelaskan informasi data sebesar **...**. 🚀

#### 🏷️ 5. Melabeli Dataframe dengan Hasil K-Medoids

Terakhir, kita tambahkan hasil label cluster ke dataframe asli untuk keperluan **profiling**.

In [ ]:
# assign label from K-medoids with 8 clusters
well_2527_copy

#### 🧩 *6. Cluster Profiling*

Tahap terakhir dari proses *clustering* adalah **profiling**, yaitu memahami karakteristik dari masing-masing cluster yang sudah terbentuk. Tujuannya:

* 🔎 Mengetahui apa yang membedakan tiap cluster
* 📝 Memberi interpretasi praktis agar hasil clustering bisa dipakai dalam pengambilan keputusan

👉 Dalam konteks ini, kita akan menganalisis cluster berdasarkan **kedalaman (DEPTH_MD)** dan **porositas neutron (NPHI)** yang diperoleh dari data logging sumur.

##### ⛏️ Profiling Berdasarkan Kedalaman (DEPTH_MD)

Kedalaman (`DEPTH_MD`) penting untuk melihat di lapisan mana masing-masing cluster terbentuk.

In [ ]:
# Profiling dari kedalaman
well_2527_copy

🔎 Urutan median kedalaman tiap cluster:

* **Cluster 0** → ......
* **Cluster 1** → ......
* **Cluster 2** → ......
* **Cluster 3** → ......
* **Cluster 4** → ......
* **Cluster 5** → ......
* **Cluster 6** → ......
* **Cluster 7** → ......

👉 Artinya: cluster membagi data sumur dari .....**.

##### 🛢️ Profiling Berdasarkan Porositas Neutron (NPHI)

Selanjutnya, mari kita analisis cluster berdasarkan `NPHI` (Neutron Porosity). Variabel ini penting karena menunjukkan **kandungan hidrogen**, yang sering dipakai untuk menilai potensi reservoir minyak atau gas.

In [ ]:
# Profiling berdasarkan NPHI
well_2527_copy

🔎 Urutan median porositas tiap cluster:

* **Cluster 0** → ......
* **Cluster 1** → ......
* **Cluster 2** → ......
* **Cluster 3** → ......
* **Cluster 4** → ......
* **Cluster 5** → ......
* **Cluster 6** → ......
* **Cluster 7** → ......

👉 Artinya: ada pola menarik bahwa .....

# Glossary

- *Variance*: Ukuran seberapa jauh sebuah kumpulan bilangan tersebar. Nilai variance yang mendekati 0 menunjukkan data tidak banyak beragam. 
- *Std (Standard Deviation)*: Akar kuadrat dari variance, dapat digunakan untuk menentukan keragaman data. Semakin kecil nilai *Std* maka data semakin tidak beragam dan sebaliknya.
- *Covariance*: Nilai yang menunjukkan hubungan antara variansi pada variable $X$ dan variansi pada variable $Y$. Tidak dapat di interpretasikan, dikarenakan memiliki nilai negatif tak hingga sampai positif tak hingga.
- *Correlation*: Nilai yang menunjukkan hubungan antar variable $X$ dengan variable $Y$. Memiliki rentang nilai -1 sampai 1. Nilai korelasi mendekati 1 menujukkan variable $X$ dan $Y$ memiliki hubungan positif kuat, sedangkan nilai korelasi mendekati -1 menunjukkan variable $X$ dan $Y$ memiliki hubungan negatif kuat.

# Referensi

- [In Depth: Principal Component Analysis (Jake VanderPlas)](https://jakevdp.github.io/PythonDataScienceHandbook/05.09-principal-component-analysis.html)
- [Intuisi PCA](https://stats.stackexchange.com/questions/2691/making-sense-of-principal-component-analysis-eigenvectors-eigenvalues/2700#2700)
- [PCA (chapter 3)](https://ourarchive.otago.ac.nz/bitstream/handle/10523/7534/OUCS-2002-12.pdf?sequence=1&isAllowed=y)
- [The Importance of Feature Scaling before PCA](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html)